# 3장. 대화 이어가기와 실행 상태 관리

같은 Session에서 전체 매출 분석과 일별 매출 분석을 이어갑니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 동일 Session·서로 다른 Turn·저장 이력 |
| 이번 실습 조합 | 새 Session에서 두 요청 수행 · 메시지 입력 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

2장의 기본 실행에 **같은 Session으로 후속 입력을 보내는 단계**를 추가합니다.

| 단계 | 직접 사용하는 API 또는 값 | 확인할 내용 |
| --- | --- | --- |
| 첫 분석 | `sessions.create()` | 전체 매출 JSON과 첫 Turn 완료 |
| 후속 분석 | `sessions.stream(session_id, input=...)` | 기존 파일을 사용한 일별 매출 JSON |
| 연속성과 이력 | Session·Turn ID, `turns.list()`, `items.list()` | 같은 Session·서로 다른 Turn과 저장된 작업 |
| 정리 | 정리 helper의 삭제·재조회 | 생성한 세션의 API상 부재 |

실시간 텍스트와 저장된 이력을 직접 확인하며, 파일 업로드·다운로드는 하지 않습니다. 수동 실습은 아래 셀을 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_intro.py --chapter 3 --live`로 실행합니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실습 준비와 실행 모드

로컬에서는 저장소 루트에서 `uv sync --locked`, `uv run jupyter lab`을 실행하고 `.venv` 커널을 선택하세요.
Colab에서는 다운로드 안내의 독립 실행본을 업로드하세요. 준비 셀이 패키지를 직접 설치하고 포함된 코드·데이터를 생성합니다.
실제 API 실습은 **RUN_LIVE=True**로 명시적으로 허용합니다.
키는 `.env.local` 또는 Colab Secrets에서 읽으며 출력하지 않습니다. 라이브 실행에는 비용이 발생합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 3
CHAPTER = "03_sessions"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print(
    {
        "chapter": NUMBER,
        "live_enabled": settings.enabled,
        "model": settings.model or "Offline mode",
    }
)


## 1. 하나의 Session, 서로 다른 두 요청

| 구분 | 첫 번째 요청 | 후속 요청 |
| --- | --- | --- |
| API | `sessions.create()` | `sessions.stream(session_id, input=...)` |
| 입력 | CSV 원문과 전체 매출 분석 지시 | 기존 파일로 일별 매출을 계산하라는 지시 |
| Session | 새로 생성 | 첫 번째 Session 사용 |
| Turn | 첫 작업 | 새로운 작업 |
| 파일 | `/workspace/sales.csv`에 저장 | 같은 파일 읽기 |

이 예제는 대화 문맥뿐 아니라 같은 환경의 파일을 다시 사용합니다. 후속 요청에서 기존 파일 경로를 명시하며 CSV 원문을 다시 보내지 않습니다. 새 세션에서도 파일이 자동으로 공유된다는 의미는 아닙니다.

먼저 로컬 CSV를 표시합니다. 이 단계는 API 호출이나 분석 실행이 아닙니다.


In [ ]:
csv_text = (settings.root / "data/sales.csv").read_text()
print(csv_text)


### 두 입력 구성하기

Agent 지침과 환경은 첫 세션 생성에서 정하고, 후속 요청에는 새 입력만 전달합니다. 첫 입력은 `METRICS_CONTRACT`로 최종 JSON 형식을 지정합니다. 후속 입력은 날짜별 매출과 최고 매출일을 요청합니다. 아래 정의만 실행해도 API 호출은 발생하지 않습니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)
task = (
    "Save the CSV below as /workspace/sales.csv. "
    "Write and run a Python standard-library analysis script, inspect stdout, "
    "and calculate the summary. "
    + METRICS_CONTRACT
    + "\nCSV input:\n"
    + csv_text
)


In [ ]:
follow_up = (
    "Read the existing /workspace/sales.csv from "
    "our previous turn. Use Python to sum quantity"
    " * unit_price by date. Return exactly a JSON "
    "object with daily_revenue mapping each date "
    "to integer revenue, and highest_revenue_date "
    "as a date string."
)


## 2. 첫 분석 실행과 완료 확인

첫 Turn의 전체 매출 결과를 확인한 다음 후속 입력을 보냅니다. 아직 작업 중일 때 입력하면 진행 중인 Turn을 조향할 수 있으므로, 이 실습은 완료를 확인하고 다음 단계로 넘어갑니다.

이벤트의 텍스트 delta를 표시하면서 `with_result_collection()`으로 최종 응답도 수집합니다. `get_final_result()`로 받은 결과와 `turns.retrieve()`로 조회한 root Turn이 모두 완료인지 확인하고, 분석 JSON을 표시합니다. JSON 파싱 성공과 계산 정확성은 다른 확인입니다.

각 스트림은 해당 셀에서 열고 닫습니다. 스트림 종료는 작업 취소나 세션 삭제가 아닙니다. 정상 경로에서는 **5절 정리 셀까지 실행**하고, 셀 사이에서 학습을 중단할 때도 정리 셀을 실행하세요. 각 단계의 `cleanup_on_error()`는 오류·일반적인 인터럽트에서 정리를 시도합니다. 강제 커널 종료에서는 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

`output_dir`는 `creation.json` 등 복구용 기록을 저장하는 로컬 경로입니다. 상태 준비 셀은 새 실행을 시작할 때 한 번만 실행하세요. 진행 중 다시 실행하면 Session ID와 제출 시도 기록을 잃을 수 있습니다.

후속 입력용 `message_key`는 여기서 한 번만 만듭니다. 같은 논리적 메시지의 재시도에는 같은 키를 사용하고, 새 메시지에는 새 키를 사용합니다. 이 예제는 재시도를 자동화하지 않으며, 후속 입력 셀 재실행으로 중복 제출하지 않도록 시도 여부를 기록합니다.

각 Turn의 타이머는 제출 직전에 시작합니다. 600초 제한은 이벤트가 도착할 때 확인하는 방식이고, 무응답 중 강제 중단하는 전체 타이머는 아닙니다. HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import json
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/intro" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
first = second = first_turn = second_turn = None
message_key = uuid4().hex
follow_up_attempted = False


### 첫 분석 실행하기

`agent.session.created`에서 세션을 기록하고 텍스트 delta를 표시합니다. 예상하지 않은 외부 작업 요청은 오류로 처리하고, `check_event()`로 실패·취소를 감지합니다. 이벤트 개수·순서를 고정하지 않으며 `idle`만 보고 성공으로 판단하지 않습니다.

아래 셀은 새 세션을 생성하므로 중복 실행하지 마세요. 출력의 Session ID·첫 Turn ID·완료 상태·전체 매출 JSON을 확인한 뒤 다음 절로 이동하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment={
            "type": "openai_hosted",
            "container_size": "small",
            "network": {"access": "disabled"},
        },
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        first = stream.get_final_result()
        sessions.setdefault(first.session_id, {"deleted": False})

    first_turn = client.beta.agents.sessions.turns.retrieve(
        first.turn_id, session_id=first.session_id, timeout=15
    )
    if first.turn.status != "completed" or first_turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nFirst analysis:")
    print({"session_id": first.session_id, "turn_id": first.turn_id,
           "status": first_turn.status})
    print(json.dumps(json.loads(first.output_text), indent=2))


## 3. 같은 Session에 후속 입력 보내기

첫 Turn이 완료됐으므로 `first.session_id`에 새 입력을 보냅니다. `sessions.stream()`은 이벤트 구독을 연결한 뒤 입력을 제출합니다. 이 helper가 실행되는 동안 다른 클라이언트가 같은 세션에 입력을 보내지 않도록 하세요.

후속 요청에는 CSV·Agent·환경 설정을 다시 전달하지 않습니다. 기존 `/workspace/sales.csv`로 날짜별 매출을 계산하도록 요청하며, 이 셀은 새로운 Turn을 시작합니다. 출력의 두 번째 Turn ID와 일별 매출 JSON을 확인하세요.

실행 오류 후에는 정리 결과와 저장된 ID부터 확인하세요. 새 키를 만들어 후속 입력 셀을 반복 실행하는 것은 복구 절차가 아닙니다. 이 예제의 오류 처리는 세션 정리를 시도하므로 오류 뒤에는 상태를 확인하고 새 실습을 준비합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if first_turn is None or first_turn.status != "completed":
        raise RuntimeError("Complete the first analysis before sending follow-up input.")
    if follow_up_attempted:
        raise RuntimeError("Follow-up submission was already attempted. Inspect saved state first.")
    follow_up_attempted = True
    started = time.monotonic()
    with client.beta.agents.sessions.stream(
        first.session_id, input=follow_up, idempotency_key=message_key
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        second = stream.get_final_result()

    second_turn = client.beta.agents.sessions.turns.retrieve(
        second.turn_id, session_id=second.session_id, timeout=15
    )
    if second.turn.status != "completed" or second_turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nFollow-up analysis:")
    print({"session_id": second.session_id, "turn_id": second.turn_id,
           "status": second_turn.status})
    print(json.dumps(json.loads(second.output_text), indent=2))


## 4. 대화 연속성과 저장된 이력 확인하기

이제 두 결과의 Session ID는 같고 Turn ID는 다른지 직접 확인합니다. 이 비교는 대화 연속성 확인이며 계산값의 정답 검사는 아닙니다.

| 대상 | 이 장에서 확인하는 목적 |
| --- | --- |
| Event | 실행 중 변화 관찰 |
| Turn | 입력으로 시작한 작업의 상태 확인 |
| Item | 명령 실행 등 저장된 작업 내용 조회 |

아래 SDK 반복은 페이지를 이어 읽습니다. 한 페이지에 모든 결과가 있다고 가정하지 않습니다. 전체 객체 대신 Turn의 ID·상태와 Item의 종류·관련 Turn ID를 표시합니다. 첫 요청과 후속 요청의 Turn이 저장된 목록에 모두 있는지도 확인합니다.

스트림은 놓친 이벤트를 재생하지 않습니다. [14장 관찰자 재연결](14_operations.ipynb)에서 입력을 재전송하지 않고 같은 Turn의 저장된 결과를 확인하는 방식을 배웁니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if second_turn is None or second_turn.status != "completed":
        raise RuntimeError("Complete the follow-up analysis before inspecting history.")
    same_session = first.session_id == second.session_id
    distinct_turns = first.turn_id != second.turn_id
    if not same_session or not distinct_turns:
        raise RuntimeError("Expected one session with two distinct turns")
    saved_turns = list(client.beta.agents.sessions.turns.list(
        first.session_id, order="asc", timeout=15,
    ))
    items = list(client.beta.agents.sessions.items.list(
        first.session_id, order="asc", timeout=15,
    ))
    if not {first.turn_id, second.turn_id} <= {t.id for t in saved_turns}:
        raise RuntimeError("Both turns were not found in stored history")
    print({"same_session": same_session, "distinct_turns": distinct_turns})
    print("Stored turns:", [{"id": t.id, "status": t.status} for t in saved_turns])
    print("Stored items:", [
        {"type": item.type, "turn_id": getattr(item, "turn_id", None)} for item in items
    ])


## 5. 세션 삭제와 정리 확인하기

두 결과와 이력을 로컬 변수에 확보했으므로 세션을 정리합니다. 아래 셀은 분석을 다시 실행하지 않습니다. 중간에 학습을 멈추는 경우에도 실행하세요.

| 동작 | 의미 |
| --- | --- |
| 스트림 종료 | 관찰 연결 닫기. 작업 취소나 세션 삭제와 별개 |
| 작업 취소 | 진행 중 작업 중단 요청. 세션 삭제와 별개 |
| 세션 삭제 | 세션을 제거하고 이후 서버 조회에서 사용하지 않음 |

`cleanup_resources()`는 이번 실행의 metadata로 생성 응답을 놓친 세션도 찾고, 필요하면 취소한 뒤 삭제·재조회로 API상 부재를 확인합니다. 출력의 `deleted`·`verified_absent`를 확인하세요. 물리적 sandbox 정리가 동기적으로 끝났다는 의미는 아닙니다.

원래 오류가 있으면 정리 오류가 그 원인을 가리지 않도록 처리합니다. 검증 도구는 삭제 전에 명령·정답 검사를 추가하며 실패해도 정리를 시도합니다. 로컬의 `first`·`second`·이력은 삭제 후에도 읽을 수 있습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 6. 기대 결과·실패 시 조치·다음 장

수동 실습에서는 첫 분석의 총매출 127500, 일별 매출 2026-01-01=65000·01-02=25000·01-03=37500, 최고 매출일 2026-01-01을 비교하세요. 이 기준값은 API 입력에 포함하지 않습니다. 계산 정확성과 성공한 Python 명령은 별도 검증기가 대조합니다.

- 두 결과의 Session ID는 같고 Turn ID는 다릅니다. 첫 Turn과 후속 Turn이 저장 이력에 모두 나타나야 합니다.
- 401/403은 인증·프로젝트 권한, 429는 API 제한을 확인합니다. 키나 원본 오류 body는 출력하지 않습니다.
- 연결 종료 후에는 같은 입력을 무조건 재전송하지 말고 정리 결과·저장된 ID를 확인합니다. 강제 종료 복구에는 `outputs/intro/<chapter>/<run_id>/creation.json`을 사용합니다. 검증 도구 실행 시 로컬 `summary.json`도 참고하세요. 자세한 절차는 [빠른 시작](../quickstart.ipynb)의 실패·중단 후 다음 행동를 참고하세요.

**선택 참고 — 취소:** `sessions.events.create(session_id, events=[{"type":"agent.session.input.cancel"}])`은 진행 중 작업의 취소 요청입니다. 기본 실습은 두 Turn 완료 후 세션을 정리하므로 취소 실습을 추가 실행하지 않습니다.

이 장은 같은 Session에서 대화와 파일을 이어 사용했습니다. [4장](04_agent_configuration.ipynb)에서는 Agent 설정을 재사용하되 각 Session의 대화와 작업 공간은 독립적이라는 차이를 배웁니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Run and continue sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions) — 후속 Turn·입력 중복 방지·취소·저장된 작업 조회.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.
- [Manage sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions/manage) — 현재 상태·pending action 확인과 세션 삭제.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
